In [2]:
import json
import itertools

In [3]:
import numpy as np 
import matplotlib.pyplot as plt

from fintorch.exchange import ONLINE_EXCHANGE
from fintorch.utils.args import DefaultArgumentParser

In [4]:
string_args = [
    "--start-date", "2024-01-01",
    "--stop-date", "2024-06-01",
]

args = DefaultArgumentParser.parse(args=string_args)

In [5]:
sdf = ONLINE_EXCHANGE.future.data.get_candles_dataframe(symbol=args.symbol, time_frame=args.time_frame)
sdf

,open,high,low,close,volume,trade
timestamp,,,,,,
1569456000,8428.15,8464.40,7700.67,8060.15,60065.945,449.0
1569542400,8061.98,8260.00,7851.94,8176.52,40034.653,504.0
1569628800,8178.26,8311.00,8001.14,8195.38,33820.919,756.0
1569715200,8199.00,8226.35,7888.98,8041.96,35966.056,1138.0
1569801600,8042.08,8331.89,7709.01,8285.84,54412.520,751.0
...,...,...,...,...,...,...
1709424000,62039.90,63330.90,61339.50,63178.50,200312.503,22965.0
1709510400,63178.50,68600.00,62299.00,68296.40,632882.144,58113.0
1709596800,68296.50,69350.00,59112.00,63747.40,1053025.132,92467.0


In [9]:
BACKWARD = 20
FORWARD = 20

df = sdf.copy()
df["bmin"] = df.close.rolling(BACKWARD).min()
df["bmax"] = df.close.rolling(BACKWARD).max()
df["fmin"] = df.close.rolling(FORWARD).min().shift(-FORWARD + 1)
df["fmax"] = df.close.rolling(FORWARD).max().shift(-FORWARD + 1)
df["up"] = df.bmin <= df.fmin
df["down"] = df.fmax <= df.bmax
df["trend"] = np.where(df.up, 1, np.where(df.down, -1, 0))
df = df.dropna()

print(df.trend.value_counts())

df

trend
 1    939
-1    508
 0    140
Name: count, dtype: int64


,open,high,low,close,volume,trade,bmin,bmax,fmin,fmax,up,down,trend
timestamp,,,,,,,,,,,,,
1571097600,8346.31,8408.62,8080.00,8160.01,86246.749,989.0,7852.79,8559.77,7411.35,9533.32,False,False,0
1571184000,8160.55,8184.00,7900.24,7992.21,70434.452,1083.0,7852.79,8559.77,7411.35,9533.32,False,False,0
1571270400,7994.76,8124.63,7930.00,8069.20,58354.602,734.0,7852.79,8559.77,7411.35,9533.32,False,False,0
1571356800,8069.41,8117.30,7812.00,7943.94,75251.352,743.0,7852.79,8559.77,7411.35,9533.32,False,False,0
1571443200,7945.57,8108.54,7852.12,7947.13,78024.395,1244.0,7852.79,8559.77,7411.35,9533.32,False,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1707782400,49943.60,50428.80,48301.00,49696.40,382348.085,36867.0,39953.30,49943.60,49696.40,63178.50,True,False,1
1707868800,49696.40,52100.00,49215.70,51823.20,395191.791,34709.0,41806.40,51823.20,50771.90,68296.40,True,False,1
1707955200,51823.10,52859.00,51334.90,51898.50,392724.603,29973.0,42021.20,51898.50,50771.90,68296.40,True,False,1
